# 02 - Bronze Ingestion

This notebook ingests the raw CSV from S3 and writes the Bronze layer as Delta format on S3.

In [ ]:
%%configure -f
{
  "conf": {
    "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension",
    "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"
  }
}

In [ ]:
from datetime import datetime

RAW_FILE_PATH = "s3://loanshield-raw/accepted_2007_to_2018Q4.csv"
BRONZE_PATH = "s3://loanshield-bronze/"
SAMPLE_ROWS = 10000  # Use None for final full-data portfolio run

start_time = datetime.now()
df_raw = spark.read.csv(RAW_FILE_PATH, header=True, inferSchema=True)
if SAMPLE_ROWS:
    df_raw = df_raw.limit(SAMPLE_ROWS)

bronze_count = df_raw.count()
df_raw.write.format("delta").mode("overwrite").save(BRONZE_PATH)

print(f"Bronze records written: {bronze_count:,}")
print(f"Bronze output path: {BRONZE_PATH}")
print(f"Duration: {datetime.now() - start_time}")

In [ ]:
df_bronze = spark.read.format("delta").load(BRONZE_PATH)
print(f"Verified Bronze records: {df_bronze.count():,}")
df_bronze.limit(10).show(truncate=False)